# EDA — Avis clients Amazon Polarity

**Objectif** : comprendre les données avant de modéliser. À la fin de ce notebook, tu dois
savoir à quoi ressemblent les avis, quels problèmes de qualité ils posent et quelles pistes
suivre pour la suite (nettoyage, modèle de base, RAG).

**Avant de commencer**

1. Télécharge l'échantillon : `uv run python -m review_pilot.dataset` (depuis la racine du projet).
2. Dans VS Code, choisis le noyau (*kernel*) `.venv` en haut à droite du notebook.

**Méthode** : pour chaque section, lis les questions, écris ton code dans les cellules vides,
puis note ce que tu observes dans une cellule Markdown. Les indices sont repliés : ne les
ouvre que si tu bloques.

> Règle d'or : on explore le **train** uniquement. Le **test** sert à évaluer les modèles ;
> le regarder en détail maintenant, c'est risquer d'orienter tes choix avec des données
> qui doivent rester « jamais vues ».

## 1. Imports et configuration

- De quelles librairies as-tu besoin pour charger, manipuler et visualiser les données ?
- Veux-tu fixer un style de graphique et une taille de figure par défaut pour tout le notebook ?

<details>
<summary>💡 Indice</summary>

`pandas`, `matplotlib.pyplot`, `seaborn`. Pour le style : `sns.set_theme()`.

</details>

## 2. Chargement

- Charge `data/raw/train.parquet` dans un DataFrame.
- Charge aussi `data/raw/test.parquet`, mais uniquement pour vérifier sa forme et ses colonnes.

<details>
<summary>💡 Indice</summary>

`pd.read_parquet("../data/raw/train.parquet")` — attention, le notebook s'exécute depuis `notebooks/`.

</details>

## 3. Premier aperçu

- Combien de lignes et de colonnes ? Est-ce cohérent avec [data/README.md](../data/README.md) ?
- Quels sont les types de chaque colonne ? Sont-ils ceux attendus ?
- À quoi ressemblent les 5 premières lignes ? Et 5 lignes tirées au hasard ?
- Combien de mémoire le DataFrame occupe-t-il ?

<details>
<summary>💡 Indice</summary>

`df.shape`, `df.dtypes`, `df.head()`, `df.sample(5, random_state=0)`, `df.info(memory_usage="deep")`.

</details>

## 4. Qualité des données

- Y a-t-il des valeurs manquantes ? Dans quelles colonnes ?
- Y a-t-il des textes vides ou composés uniquement d'espaces ?
- Y a-t-il des lignes entièrement dupliquées ? Des `content` identiques avec des titres différents ?
  Des `content` identiques avec des **labels différents** (le pire cas) ?
- Des avis du train se retrouvent-ils dans le test ? Si oui, c'est une **fuite de données**
  (*data leakage*) : le modèle serait évalué sur des exemples qu'il a déjà vus, et son score
  serait artificiellement gonflé.

<details>
<summary>💡 Indice</summary>

`df.isna().sum()`, `df["content"].str.strip().eq("").sum()`, `df.duplicated().sum()`, `df.duplicated(subset="content")`, `train["content"].isin(test["content"]).sum()`.

</details>

## 5. Variable cible : le label

- Quelle est la répartition des labels (en nombre et en pourcentage) ?
- Les classes sont-elles équilibrées ? Qu'est-ce que ça implique pour choisir une métrique
  (accuracy, F1…) plus tard ?
- Visualise cette répartition avec un graphique adapté.

<details>
<summary>💡 Indice</summary>

`df["label"].value_counts(normalize=True)`, `sns.countplot(data=df, x="label")`.

</details>

## 6. Longueur des textes

- Combien de caractères et de mots dans `title` et dans `content` ? (min, max, moyenne, médiane)
- Trace la distribution des longueurs de `content`. Est-elle symétrique ? Y a-t-il un plafond
  suspect (textes tronqués à une longueur fixe) ?
- Les avis négatifs sont-ils plus longs ou plus courts que les positifs ?
- Regarde les avis les plus courts et les plus longs : qu'ont-ils de particulier ?

<details>
<summary>💡 Indice</summary>

`df["content"].str.len()`, `df["content"].str.split().str.len()`, `.describe()`, `sns.histplot(data=df, x=..., hue="label")`, `sns.boxplot(...)`.

</details>

## 7. Variables dérivées et corrélations

Une matrice de corrélation ne s'applique qu'à des colonnes numériques : il faut d'abord
**fabriquer** des variables à partir du texte.

- Crée quelques variables : longueur en caractères, nombre de mots, nombre de `!`,
  nombre de `?`, part de majuscules, longueur du titre… Invente-en d'autres !
- Calcule la matrice de corrélation entre ces variables et le label, puis affiche-la en heatmap.
- Quelles variables semblent liées au label ? Lesquelles sont redondantes entre elles ?
- Attention : corrélation n'est pas causalité, et une corrélation faible n'implique pas
  qu'une variable soit inutile (la relation peut être non linéaire).

<details>
<summary>💡 Indice</summary>

`df["content"].str.count("!")`, `df.select_dtypes("number").corr()`, `sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)`.

</details>

## 8. Vocabulaire

- Quels sont les mots les plus fréquents, tous avis confondus ? Sont-ils informatifs ?
- Et si tu retires les mots vides (*stopwords* : « the », « and », « is »…) ?
- Quels mots sont les plus fréquents dans les avis négatifs ? Dans les positifs ?
- Que deviennent les négations (« not good ») si on découpe mot par mot ?

<details>
<summary>💡 Indice</summary>

`collections.Counter`, `str.lower()`, `str.split()`. Liste de stopwords : `sklearn.feature_extraction.text.ENGLISH_STOP_WORDS` (il faudra ajouter scikit-learn) ou une petite liste faite main.

</details>

## 9. Lecture d'exemples

Les statistiques ne remplacent pas la lecture : lis vraiment des avis.

- Lis 5 avis positifs et 5 négatifs tirés au hasard. Le label te semble-t-il juste ?
- Repère le bruit : séquences `\n` littérales, guillemets doublés, HTML, fautes, autres langues…
- Trouve des cas ambigus (ironie, avis mitigé) : comment un modèle pourrait-il les traiter ?

<details>
<summary>💡 Indice</summary>

`df.query("label == 0").sample(5, random_state=1)`, et `print()` sur le texte pour le lire en entier.

</details>

## 10. Conclusions

Résume en quelques puces ce que tu as appris :

- **Qualité** : quels problèmes corriger dans `data/processed/` (doublons, bruit, textes vides…) ?
- **Cible** : équilibre des classes, métrique à privilégier.
- **Texte** : longueurs, vocabulaire, bruit — quel nettoyage avant la modélisation ?
- **Pistes** : quelles variables ou idées tester pour le premier modèle ? Et pour le RAG ?